In [1]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
from sklearn.svm import LinearSVC



In [2]:
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

train_df1 = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv"
)



/kaggle/input/description.md
/kaggle/input/sample_submission.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/test.csv
/kaggle/input/train.csv.zip
/kaggle/input/train.csv
/kaggle/input/test.csv.zip
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/description.md
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv.zip
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv.zip
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv.zip


In [3]:
print(train_df1["score"].dtypes)



int64


In [4]:
print(train_df1.head(10))



  essay_id                                          full_text  score
0  663d2cf  Dear State Senator,\n\nI am arguing in favor o...      3
1  3a20bfb  In " The Challenge of Exploring Venus" The aut...      2
2  6adae64  Teachers can have a hard time telling if their...      3
3  c81ccdf  Using dirverless cars can be very dangrous in ...      3
4  9549d7f  In " The Challenge of Exploring Venus" the aut...      3
5  15992f4  What would the world be like if we could just ...      3
6  1b7e42c  Alien Landform?\n\nDo you think that the face ...      1
7  e06d5d9  My argument is for "Driverless cars", the reas...      3
8  a779bc6  Have you ever seen those movies will the smart...      4
9  ca7b3c6  Dont you get tired of driving sometimes, Get s...      2


In [5]:
print(train_df1["score"].value_counts())



score
3    5629
2    4249
4    3563
1    1124
5     876
6     135
Name: count, dtype: int64


In [6]:
submission_template = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv"
)
print(submission_template.head())



  essay_id  score
0  d550b2d      4
1  0c10954      5
2  ef04816      3
3  88ab8d9      5
4  cee8c0f      5


In [7]:
cList = {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how'll": "how will",
    "how's": "how is",
    "I'd": "I would",
    "I'd've": "I would have",
    "I'll": "I will",
    "I'll've": "I will have",
    "I'm": "I am",
    "I've": "I have",
    "isn't": "is not",
    "it'd": "it had",
    "it'd've": "it would have",
    "it'll": "it will",
    "it'll've": "it will have",
    "it's": "it is",
    "let's": "let us",
    "ma'am": "madam",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "o'clock": "of the clock",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shan't've": "shall not have",
    "she'd": "she would",
    "she'd've": "she would have",
    "she'll": "she will",
    "she'll've": "she will have",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so is",
    "that'd": "that would",
    "that'd've": "that would have",
    "that's": "that is",
    "there'd": "there had",
    "there'd've": "there would have",
    "there's": "there is",
    "they'd": "they would",
    "they'd've": "they would have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they're": "they are",
    "they've": "they have",
    "to've": "to have",
    "wasn't": "was not",
    "we'd": "we had",
    "we'd've": "we would have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what'll've": "what will have",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "when's": "when is",
    "when've": "when have",
    "where'd": "where did",
    "where's": "where is",
    "where've": "where have",
    "who'll": "who will",
    "who'll've": "who will have",
    "who's": "who is",
    "who've": "who have",
    "why's": "why is",
    "why've": "why have",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'alls": "you alls",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "you'd": "you had",
    "you'd've": "you would have",
    "you'll": "you you will",
    "you'll've": "you you will have",
    "you're": "you are",
    "you've": "you have",
}



In [8]:
c_re = re.compile("(%s)" % "|".join(cList.keys()))




In [9]:
def expandContractions(text, c_re=c_re):
    def replace(match):
        return cList[match.group(0)]

    return c_re.sub(replace, text)




In [10]:
def removeHTML(x):
    html = re.compile(r"<.*?>")
    return html.sub(r"", x)




In [11]:
def dataPreprocessing(x):
    x = x.apply(lambda s: s.lower())
    x = x.apply(removeHTML)
    x = x.apply(lambda s: re.sub("@\w+", "", s))
    x = x.apply(lambda s: re.sub("'\d+", "", s))
    x = x.apply(lambda s: re.sub("\d+", "", s))
    x = x.apply(lambda s: re.sub("http\w+", "", s))
    x = x.apply(lambda s: re.sub(r"\s+", " ", s))
    x = x.apply(expandContractions)
    x = x.apply(lambda s: re.sub(r"\.+", ".", s))
    x = x.apply(lambda s: re.sub(r"\,+", ",", s))
    x = x.apply(lambda s: re.sub("\n", "", s))
    x = x.apply(lambda s: re.sub("[^\w\s]", "", s))
    x = x.apply(lambda s: s.strip())
    return x




In [12]:
x = dataPreprocessing(train_df1["full_text"])
print(x.head())



0    dear state senator i am arguing in favor of ch...
1    in  the challenge of exploring venus the autho...
2    teachers can have a hard time telling if their...
3    using dirverless cars can be very dangrous in ...
4    in  the challenge of exploring venus the autho...
Name: full_text, dtype: object


In [13]:
test_df1 = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv"
)



In [14]:
x0 = dataPreprocessing(test_df1["full_text"])



In [15]:
y = train_df1["score"]  # use the single score column as target



In [16]:
# use stratified split to preserve class distribution, which gives a more reliable validation kappa
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.1, random_state=123, stratify=y
)



In [17]:
print(
    "Shapes -> X_train:",
    X_train.shape,
    "y_train:",
    y_train.shape,
    "X_test:",
    X_test.shape,
    "y_test:",
    y_test.shape,
)



Shapes -> X_train: (14018,) y_train: (14018,) X_test: (1558,) y_test: (1558,)


In [18]:
# Removed stop‑word filtering to retain more signal and kept other TF‑IDF settings unchanged
text_vectorizer = TfidfVectorizer(
    stop_words=None,
    sublinear_tf=True,
    strip_accents="unicode",
    binary=False,
    analyzer="word",
    token_pattern=r"\w{2,}",
    ngram_range=(1, 2),  # capture unigrams and bigrams
    norm="l2",
    use_idf=True,
    smooth_idf=False,
    max_features=800000,
    min_df=2,
)



In [19]:
X_train_features = text_vectorizer.fit_transform(X_train)



In [20]:
X_test_features = text_vectorizer.transform(X_test)
# Slightly larger C gives the classifier a bit more flexibility
clf = LinearSVC(C=2.0, random_state=123, dual=False, class_weight="balanced")
clf.fit(X_train_features, y_train.values.ravel())
y_pred = clf.predict(X_test_features)



In [21]:
print(confusion_matrix(y_test.values.ravel(), y_pred.ravel()))



[[ 49  49  13   1   0   0]
 [ 28 218 145  34   0   0]
 [ 12 133 282 131   5   0]
 [  2  17 130 195  12   0]
 [  0   0   8  63  17   0]
 [  0   0   0  10   4   0]]


In [22]:
print(classification_report(y_test.values.ravel(), y_pred.ravel()))



              precision    recall  f1-score   support

           1       0.54      0.44      0.48       112
           2       0.52      0.51      0.52       425
           3       0.49      0.50      0.49       563
           4       0.45      0.55      0.49       356
           5       0.45      0.19      0.27        88
           6       0.00      0.00      0.00        14

    accuracy                           0.49      1558
   macro avg       0.41      0.37      0.38      1558
weighted avg       0.49      0.49      0.48      1558



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [23]:
kappa = cohen_kappa_score(y_test.values.ravel(), y_pred.ravel(), weights="quadratic")
print("Cohen's kappa score:", kappa)



Cohen's kappa score: 0.6352878657236625


In [24]:
st_features = text_vectorizer.transform(x0)
test_predictions = clf.predict(st_features)



In [25]:
submission = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv"
)
submission["score"] = test_predictions
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")
display(submission.head())

Submission saved to submission.csv


,essay_id,score
0,d550b2d,2
1,0c10954,2
2,ef04816,3
3,88ab8d9,1
4,cee8c0f,2
